# Notebook 15 ? Step 2 FM sequence-length ablation

Notebook ??c l?p cho Step 2: chu?n b? token cache, geometry kh?p model/sequence length, ch?y/resume 25 t? h?p v? t?ng h?p mean/std ri?ng t?ng test set. Hyperparameter l?y t? manifest Optuna Step 1; m?i t? h?p, k? c? 1001/501, ???c train l?i v?i geometry ESM1b/NTv1 kh?p. Cache token shard ?? ho?n t?t ???c gi? nguy?n v? t?i s? d?ng.


In [ ]:
from pathlib import Path
import sys, json, os
import pandas as pd
from IPython.display import display

_cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "core").is_dir() and (p / "controllers").is_dir()), _cwd)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Same data/cache/batch locations as Notebook 14; change only if intentionally using another batch.
BASE_DIR = Path(r"F:\variant_data")
POOLED_CACHE_ROOT = BASE_DIR / "fm_embeddings" / "sequence_ablation"
TOKEN_CACHE_ROOT = BASE_DIR / "fm_embeddings" / "token_level_cache" / "module06"
FM_ABLATION_BATCH_RUN_DIR = REPO_ROOT / "experiments" / "fm_ablation_20261117_003149"
GEOMETRY_K = 64
STEP2_GEOMETRY_ROOT_NAME = "geometry_v2_k64_step2_matched"
STEP2_GEOMETRY_NORMALIZATION_ROOT = "normalization_artifacts_v2_step2_matched"
STEP2_GEOMETRY_INDEX_ROOT = "faiss_indexes_v2_step2_matched"
GEOM_FEATURE_NAMES = [f"{m}_{f}" for m in ("dna", "prot") for f in ("LVD_L2", "LVD_Cosine", "LID", "LVD_Relative")]
TARGET_CONFIGS = [{
    "name": "B_with_geometry",
    "ablation": "bio_core_dna_prot_spliceai_geom",
    "active_modalities": ["dna", "prot", "bio_core", "spliceai", "geom"],
    "geom_feature_names": GEOM_FEATURE_NAMES,
    "geometry_root": STEP2_GEOMETRY_ROOT_NAME,
}]
DATASETS = [
    {"name": "Train3Val_Test", "train": "train3", "val": "val", "test": "test"},
    {"name": "Train3Val_ClinVarHQ", "train": "train3", "val": "val", "test": "clinvarhq"},
    {"name": "Train3Val_ProteinGym", "train": "train3", "val": "val", "test": "proteingym"},
    {"name": "Train3Val_UniProt", "train": "train3", "val": "val", "test": "uniprot"},
]
PIPELINE_CONFIG = {"batch_size": 256, "epochs": 30, "lr": 1e-4, "weight_decay": 1e-3,
                   "lr_factor": 0.5, "lr_patience": 3, "early_stop_patience": 6,
                   "num_workers": 0, "seed": 42, "xgb_n_jobs": 8}
from core.module06_architecture_experiments.fm_ablation_preparation import build_split_configs
STEP2_SPLITS = build_split_configs(BASE_DIR)
STEP2_ROOT = FM_ABLATION_BATCH_RUN_DIR / "module06_architecture_experiments" / "step2_sequence_length_ablation"
print("Repository:", REPO_ROOT)
print("FM ablation batch:", FM_ABLATION_BATCH_RUN_DIR)
print("Token cache root:", TOKEN_CACHE_ROOT)
print("Geometry features:", GEOM_FEATURE_NAMES)
print("Splits:", [split["name"] for split in STEP2_SPLITS])


Repository: D:\Missense_variant_pathogenicity_prediction
FM ablation batch: D:\Missense_variant_pathogenicity_prediction\experiments\fm_ablation_20261117_003149
Token cache root: F:\variant_data\fm_embeddings\token_level_cache\module06
Geometry features: ['dna_LVD_L2', 'dna_LVD_Cosine', 'dna_LID', 'dna_LVD_Relative', 'prot_LVD_L2', 'prot_LVD_Cosine', 'prot_LID', 'prot_LVD_Relative']
Splits: ['train3', 'val', 'test', 'clinvarhq', 'proteingym', 'uniprot']


## Step 2 ? sequence-length ablation: token-cache preparation

Ch? d?ng NTv1 cho DNA v? ESM1b cho protein. Grid DNA l? 301/601/1001/2001/3501; protein l? 31/101/201/501/1001. Ri?ng DNA 3501, extractor ??c c?t chu?i 10001, c?t 3250 nucleotide m?i b?n quanh variant v? x?c nh?n variant ? gi?a c?a s? m?i (index 1750). File parquet g?c kh?ng b? s?a. Token cache resume theo shard; n?u model kh?ng h? tr? ?? 3501 nucleotide th? extraction d?ng thay v? c?t token ?m th?m.


In [ ]:
# Token extraction for the selected FM pair and all requested lengths. Resume-safe; no training in this cell.
import importlib
import os
import pandas as pd
from IPython.display import display
from core.module06_architecture_experiments.fm_ablation_preparation import (
    inspect_token_cache, extract_missing_token_caches,
)
from core.module06_architecture_experiments.fm_sequence_length_ablation import (
    DNA_LENGTHS as STEP2_DNA_LENGTHS,
    PROTEIN_LENGTHS as STEP2_PROTEIN_LENGTHS,
    SEEDS as STEP2_SEEDS,
    build_selected_model_configs, verify_sequence_columns, build_center_geometry_embeddings,
)

STEP2_TOKEN_MODELS = build_selected_model_configs(STEP2_DNA_LENGTHS, STEP2_PROTEIN_LENGTHS)
STEP2_ROOT = Path(FM_ABLATION_BATCH_RUN_DIR) / "module06_architecture_experiments" / "step2_sequence_length_ablation"
STEP2_ROOT.mkdir(parents=True, exist_ok=True)
STEP2_COLUMN_AUDIT = pd.DataFrame(verify_sequence_columns(STEP2_SPLITS, STEP2_TOKEN_MODELS))
print(f"Requested sequence-length pairs: {len(STEP2_DNA_LENGTHS) * len(STEP2_PROTEIN_LENGTHS)}")
print("DNA lengths:", STEP2_DNA_LENGTHS, "| protein lengths:", STEP2_PROTEIN_LENGTHS)
print("Selected FM models: NTv1 DNA + ESM1b protein | seeds:", STEP2_SEEDS)
display(STEP2_COLUMN_AUDIT.groupby(["Model", "Sequence_Length", "Source_Sequence_Length", "Status"], as_index=False).size())
print("DNA 3501 crop: source columns ref_seq_10001/alt_seq_10001; crop 3250 nt on each side of the centered variant; variant index 1750.")

STEP2_CACHE_BEFORE = pd.DataFrame(inspect_token_cache(TOKEN_CACHE_ROOT, STEP2_TOKEN_MODELS, STEP2_SPLITS))
print("Token cache inventory before extraction:")
display(STEP2_CACHE_BEFORE.groupby(["Model", "Status"], as_index=False).size())
os.environ["HF_HUB_OFFLINE"] = "1"
STEP2_TOKEN_CACHE_PREPARATION = extract_missing_token_caches(
    model_configs=STEP2_TOKEN_MODELS,
    split_configs=STEP2_SPLITS,
    cache_root=TOKEN_CACHE_ROOT,
    batch_run_dir=FM_ABLATION_BATCH_RUN_DIR,
    device="cuda",
    strict_context=True,  # Fail instead of silently truncating any requested full sequence.
)
STEP2_TOKEN_CACHE_INDEX = STEP2_TOKEN_CACHE_PREPARATION["index"]
STEP2_CACHE_AFTER = pd.DataFrame(inspect_token_cache(TOKEN_CACHE_ROOT, STEP2_TOKEN_MODELS, STEP2_SPLITS))
display(STEP2_CACHE_AFTER.groupby(["Model", "Status"], as_index=False).size())
if (STEP2_CACHE_AFTER["Status"] != "complete").any():
    raise RuntimeError("Step 2 token-cache extraction is incomplete; rerun this cell to resume missing shards.")
print("Step 2 token cache index:", STEP2_TOKEN_CACHE_PREPARATION["index_path"])


Requested sequence-length pairs: 25
DNA lengths: (301, 601, 1001, 2001, 3501) | protein lengths: (31, 101, 201, 501, 1001)
Selected FM models: NTv1 DNA + ESM1b protein | seeds: (5, 6, 7, 37, 42)


,Model,Sequence_Length,Source_Sequence_Length,Status,size
0,esm1b_650m_prot_1001,1001,1001,ready,6
1,esm1b_650m_prot_101,101,101,ready,6
2,esm1b_650m_prot_201,201,201,ready,6
3,esm1b_650m_prot_31,31,31,ready,6
4,esm1b_650m_prot_501,501,501,ready,6
5,nt_v1_500m_dna_1001,1001,1001,ready,6
6,nt_v1_500m_dna_2001,2001,2001,ready,6
7,nt_v1_500m_dna_301,301,301,ready,6
8,nt_v1_500m_dna_3501,3501,10001,ready,6
9,nt_v1_500m_dna_601,601,601,ready,6


DNA 3501 crop: source columns ref_seq_10001/alt_seq_10001; crop 3250 nt on each side of the centered variant; variant index 1750.
Token cache inventory before extraction:


,Model,Status,size
0,esm1b_650m_prot_1001,complete,6
1,esm1b_650m_prot_101,complete,6
2,esm1b_650m_prot_201,complete,6
3,esm1b_650m_prot_31,complete,6
4,esm1b_650m_prot_501,complete,6
5,nt_v1_500m_dna_1001,complete,6
6,nt_v1_500m_dna_2001,complete,6
7,nt_v1_500m_dna_301,complete,6
8,nt_v1_500m_dna_3501,complete,6
9,nt_v1_500m_dna_601,complete,6


c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict_clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Model,Status,size
0,esm1b_650m_prot_1001,complete,6
1,esm1b_650m_prot_101,complete,6
2,esm1b_650m_prot_201,complete,6
3,esm1b_650m_prot_31,complete,6
4,esm1b_650m_prot_501,complete,6
5,nt_v1_500m_dna_1001,complete,6
6,nt_v1_500m_dna_2001,complete,6
7,nt_v1_500m_dna_301,complete,6
8,nt_v1_500m_dna_3501,complete,6
9,nt_v1_500m_dna_601,complete,6


Step 2 token cache index: D:\Missense_variant_pathogenicity_prediction\experiments\fm_ablation_20261117_003149\module06_architecture_experiments\fm_ablation_token_cache_index.json


### Step 2 ? build model- and length-matched geometry

Geometry ???c t?nh t? center embedding ref/alt c?a ch?nh NTv1 DNA length v? ESM1b protein length t??ng ?ng. Cell d?n xu?t vector center t? token cache (kh?ng ch?y FM inference), r?i t?o LVD/LID geometry v?i k=64 v? chu?n h?a theo train3. Embedding trung gian, geometry, FAISS index v? scaler ??u ? th? m?c ri?ng; kh?ng ghi pooled cache d?ng cho Step 3.


In [ ]:
# Derive center vectors for geometry only, then build/resume all matched k=64 geometry files.
from core.module06_architecture_experiments.fm_sequence_length_ablation import build_center_geometry_embeddings
from core.module03_geometry_extraction.geometry_v2 import GeometryV2Config, run_geometry_v2
import pyarrow.parquet as pq

STEP2_GEOMETRY_EMBEDDING_ROOT = BASE_DIR / "fm_embeddings" / "step2_geometry_center"
STEP2_GEOMETRY_MODELS = [model["name"] for model in STEP2_TOKEN_MODELS]
STEP2_CENTER_GEOMETRY_EMBEDDINGS = build_center_geometry_embeddings(
    model_configs=STEP2_TOKEN_MODELS,
    split_configs=STEP2_SPLITS,
    token_cache_index=STEP2_TOKEN_CACHE_INDEX,
    output_root=STEP2_GEOMETRY_EMBEDDING_ROOT,
)
display(STEP2_CENTER_GEOMETRY_EMBEDDINGS.groupby(["Model", "Status"], as_index=False).size())

STEP2_GEOMETRY_CONFIG = GeometryV2Config(
    base_dir=str(BASE_DIR),
    embedding_dir=str(STEP2_GEOMETRY_EMBEDDING_ROOT),
    contexts=["train3"],
    splits=[split["name"] for split in STEP2_SPLITS],
    models=STEP2_GEOMETRY_MODELS,
    poolings=["center"],
    k_neighbors=GEOMETRY_K,
    resume=True,
    geometry_root=STEP2_GEOMETRY_ROOT_NAME,
    normalization_root=STEP2_GEOMETRY_NORMALIZATION_ROOT,
    index_root=STEP2_GEOMETRY_INDEX_ROOT,
)
STEP2_GEOMETRY_RUN = run_geometry_v2(STEP2_GEOMETRY_CONFIG)

STEP2_GEOMETRY_AUDIT = []
for _model in STEP2_TOKEN_MODELS:
    for _split in STEP2_SPLITS:
        _path = BASE_DIR / STEP2_GEOMETRY_ROOT_NAME / "train3" / _split["name"] / f"{_model['name']}_center_geom_v2_norm.parquet"
        if not _path.is_file():
            raise FileNotFoundError(f"Matched geometry file is missing: {_path}")
        _columns = set(pq.read_schema(_path).names)
        _required = {"Variant_ID", "LVD_L2", "LVD_Cosine", "LID", "LVD_Relative"}
        _missing = sorted(_required - _columns)
        if _missing:
            raise KeyError(f"Geometry file {_path} misses selected columns: {_missing}")
        STEP2_GEOMETRY_AUDIT.append({"Model": _model["name"], "Split": _split["name"], "Status": "ready", "Path": str(_path)})
STEP2_GEOMETRY_AUDIT = pd.DataFrame(STEP2_GEOMETRY_AUDIT)
display(STEP2_GEOMETRY_AUDIT.groupby("Model", as_index=False).agg(Splits=("Split", "nunique"), Status=("Status", "first")))
print("Matched geometry root:", BASE_DIR / STEP2_GEOMETRY_ROOT_NAME / "train3")


[GEOM EMBEDDING RESUME] nt_v1_500m_dna_301/train3
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_301/val
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_301/test
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_301/clinvarhq
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_301/proteingym
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_301/uniprot
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_601/train3
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_601/val
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_601/test
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_601/clinvarhq
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_601/proteingym
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_601/uniprot
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/train3
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/val
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/test
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/clinvarhq
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/proteingym
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/uniprot
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_2001/train3
[GEOM EMBEDDING RESUME] nt_v1_500m

,Model,Status,size
0,esm1b_650m_prot_1001,reused,6
1,esm1b_650m_prot_101,reused,6
2,esm1b_650m_prot_201,reused,6
3,esm1b_650m_prot_31,reused,6
4,esm1b_650m_prot_501,reused,6
5,nt_v1_500m_dna_1001,reused,6
6,nt_v1_500m_dna_2001,reused,6
7,nt_v1_500m_dna_301,reused,6
8,nt_v1_500m_dna_3501,reused,6
9,nt_v1_500m_dna_601,reused,6


[V2 RESUME] Skip train3/nt_v1_500m_dna_301_center
[V2 RESUME] Skip train3/nt_v1_500m_dna_601_center
[V2 RESUME] Skip train3/nt_v1_500m_dna_1001_center
[V2 RESUME] Skip train3/nt_v1_500m_dna_2001_center
[V2 RESUME] Skip train3/nt_v1_500m_dna_3501_center
[V2 RESUME] Skip train3/esm1b_650m_prot_31_center
[V2 RESUME] Skip train3/esm1b_650m_prot_101_center
[V2 RESUME] Skip train3/esm1b_650m_prot_201_center
[V2 RESUME] Skip train3/esm1b_650m_prot_501_center
[V2 RESUME] Skip train3/esm1b_650m_prot_1001_center


,Model,Splits,Status
0,esm1b_650m_prot_1001,6,ready
1,esm1b_650m_prot_101,6,ready
2,esm1b_650m_prot_201,6,ready
3,esm1b_650m_prot_31,6,ready
4,esm1b_650m_prot_501,6,ready
5,nt_v1_500m_dna_1001,6,ready
6,nt_v1_500m_dna_2001,6,ready
7,nt_v1_500m_dna_301,6,ready
8,nt_v1_500m_dna_3501,6,ready
9,nt_v1_500m_dna_601,6,ready


Matched geometry root: F:\variant_data\geometry_v2_k64_step2_matched\train3


## Step 2 ? train all 25 sequence-length pairs

M?i c?p d?ng geometry NTv1/ESM1b kh?p ??ng model v? sequence length, Cross-Attention 3 modality, c?a s? ref/alt 33 token, bio + SpliceAI, MLP v? 5 seed. Hyperparameter c?a t?ng seed l?y t? manifest Step 1. Checkpoint Step 1 ch? l? ngu?n hyperparameter; c? 25 c?p, bao g?m 1001/501, ??u ???c train l?i v? geometry ?? ??i. C?c run m?i h? tr? resume.


In [ ]:
# Run/resume all 25 pairs ? five seeds. Step 1 checkpoints supply hyperparameters only.
import importlib
import json
from core.module05v2_fusion_classifier.pipeline import FusionBatchPipeline
from core.module06_architecture_experiments.fm_sequence_length_ablation import (
    load_source_seed_configs, run_sequence_length_ablation,
)

STEP2_SOURCE_ROOT = Path(FM_ABLATION_BATCH_RUN_DIR) / "module06_architecture_experiments" / "step1_optuna_nt_v1_dna1001_esm1b_prot501_5seed"
STEP2_SOURCE_CONFIGS, STEP2_SOURCE_MANIFESTS = load_source_seed_configs(
    STEP2_SOURCE_ROOT, STEP2_SEEDS
)
STEP2_SOURCE_CONFIG_TABLE = pd.DataFrame([
    {"Seed": seed, **config, "Source_Checkpoint": STEP2_SOURCE_MANIFESTS[seed]["checkpoint"]}
    for seed, config in STEP2_SOURCE_CONFIGS.items()
]).sort_values("Seed")
display(STEP2_SOURCE_CONFIG_TABLE)

if len(TARGET_CONFIGS) != 1 or TARGET_CONFIGS[0]["name"] != "B_with_geometry":
    raise ValueError("Step 2 is fixed to B_with_geometry only.")
if set(STEP2_TOKEN_CACHE_INDEX["models"]) != {model["name"] for model in STEP2_TOKEN_MODELS}:
    raise ValueError("Step 2 token-cache index does not cover exactly the requested NTv1/ESM1b length configs.")

# Reload the shared runner so the token-only sequence-ablation loader is active.
architecture_runner_module = importlib.reload(importlib.import_module(
    "core.module06_architecture_experiments.architecture_experiment_runner"
))

def _step2_pipeline_factory(seed, dna_model, protein_model):
    return FusionBatchPipeline(
        base_dir=str(BASE_DIR),
        config={**PIPELINE_CONFIG, "seed": int(seed)},
        datasets=DATASETS,
        models_space=[dna_model, protein_model],
        pooling_strategies=["center"],
        experiments=[{"name": "PyTorch_CrossAttention_SequenceLength", "type": "pytorch", "fusion": "cross_attention"}],
        explainability={"enable_shap": False, "enable_lime": False},
        fm_profile_json=str(BASE_DIR / "profiling" / "fm_profiling_sequence_ablation.json"),
        geom_profile_json=None,
        geometry_dir=str(BASE_DIR / STEP2_GEOMETRY_ROOT_NAME / "{context}"),
        geom_feature_names=GEOM_FEATURE_NAMES,
        batch_run_dir=str(FM_ABLATION_BATCH_RUN_DIR),
        embedding_dir=str(POOLED_CACHE_ROOT),
    )

STEP2_RUN_RESULT = run_sequence_length_ablation(
    pipeline_factory=_step2_pipeline_factory,
    target_configs=TARGET_CONFIGS,
    datasets=DATASETS,
    model_configs=STEP2_TOKEN_MODELS,
    token_cache_index=STEP2_TOKEN_CACHE_INDEX,
    source_configs=STEP2_SOURCE_CONFIGS,
    source_manifests=STEP2_SOURCE_MANIFESTS,
    step_root=STEP2_ROOT,
    device="cuda",
    token_batch_size=64,
    display_fn=print,
)
print(STEP2_RUN_RESULT)


,Seed,attention_dropout,dropout,early_stop_patience,feature_dropout,ffn_multiplier,k_radius,learning_rate,lr_factor,lr_patience,max_epochs,model_dim,num_heads,num_layers,pooling,seed,weight_decay,Source_Checkpoint
0,5,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,5,0.000003,D:\Missense_variant_pathogenicity_prediction\e...
1,6,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,6,0.000003,D:\Missense_variant_pathogenicity_prediction\e...
2,7,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,7,0.000003,D:\Missense_variant_pathogenicity_prediction\e...
3,37,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,37,0.000003,D:\Missense_variant_pathogenicity_prediction\e...
4,42,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,42,0.000003,D:\Missense_variant_pathogenicity_prediction\e...


[CrossAttention_ThreeModalities_FM_SequenceLength] Preparing train dataset and token-cache loader...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.



Window cache dna/train3: 100%|██████████| 726/726 [00:21<00:00, 33.32shard/s]


[Window cache] dna/train3: 23208 samples ready in 21.8s


Window cache protein/train3: 100%|██████████| 1451/1451 [00:40<00:00, 35.56shard/s]


[Window cache] protein/train3: 23208 samples ready in 40.8s
[CrossAttention_ThreeModalities_FM_SequenceLength] Train loader ready: 363 batches in 64.4s
[CrossAttention_ThreeModalities_FM_SequenceLength] Preparing validation dataset and token-cache loader...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.



... (47 outputs left out)

Window cache protein/val: 100%|██████████| 857/857 [00:00<00:00, 896.35shard/s]


[Window cache] protein/val: 13701 samples ready in 1.0s
[CrossAttention_ThreeModalities_FM_SequenceLength] Validation loader ready: 215 batches in 2.5s


CrossAttention_ThreeModalities_FM_SequenceLength | ref_alt_concat_window_33_sequence_length_ablation | B_with_geometry:   3%|▎         | 1/30 [00:14<07:00, 14.50s/epoch, best_epoch=1, threshold=0.6033, train_loss=0.3477, val_MCC=0.7861, val_loss=0.1831]

[Epoch 1] train=11.5s; total including validation and checkpoint=14.5s


## Step 2 ? results and per-test-set mean/std

Cell t?ng h?p k?t qu? c?a c?c run geometry-matched m?i cho ?? 25 t? h?p. Ch?n c?u h?nh theo validation MCC; b?o c?o ri?ng t?ng test set, kh?ng g?p b?n test set. C?c CSV l?u metric, profiling, seed v? hyperparameter t?ng run c?ng mean/std.


In [ ]:
# Read completed geometry-matched Step 2 rows and save full and mean/std tables.
from IPython.display import display
from core.module06_architecture_experiments.fm_sequence_length_ablation import summarize_sequence_length_ablation

STEP2_SUMMARY = summarize_sequence_length_ablation(
    step_root=STEP2_ROOT,
    seeds=STEP2_SEEDS,
)
print("Validation mean/std across seeds; ranking is by validation MCC:")
display(STEP2_SUMMARY["validation_mean_std"])

STEP2_TEST_DISPLAY_COLS = [
    "DNA_Length", "Protein_Length", "N_Seeds",
    "Accuracy_mean", "Accuracy_std", "Precision_mean", "Precision_std",
    "Recall_mean", "Recall_std", "Specificity_mean", "Specificity_std",
    "F1_Score_mean", "F1_Score_std", "MCC_mean", "MCC_std",
    "AUROC_mean", "AUROC_std", "AUPRC_mean", "AUPRC_std",
    "FM_GFLOPs/Sample_mean", "FM_Latency_ms/Sample_mean",
    "Geom_GFLOPs/Sample_mean", "Geom_Latency_ms/Sample_mean",
    "Classifier_GFLOPs/Sample_mean", "Classifier_Latency_ms/Sample_mean",
    "Classifier_Total_Params_mean", "Classifier_Param_Mem_MB_mean",
    "E2E_GFLOPs/Sample_mean", "E2E_Latency_ms/Sample_mean",
    "E2E_Total_Params_mean", "E2E_Param_Mem_MB_mean", "Max_Peak_VRAM_MB_mean",
]
for _dataset_name, _dataset_frame in STEP2_SUMMARY["test_mean_std_by_dataset"].groupby("Dataset", sort=False):
    _columns = [column for column in STEP2_TEST_DISPLAY_COLS if column in _dataset_frame.columns]
    print(f"\nTest set: {_dataset_name} (sorted by mean MCC)")
    display(_dataset_frame[_columns].sort_values("MCC_mean", ascending=False).reset_index(drop=True))

print("Combo completion inventory:")
display(STEP2_SUMMARY["inventory"].sort_values(["DNA_Length", "Protein_Length"]).reset_index(drop=True))
print("Saved detailed and aggregate CSVs under:", STEP2_SUMMARY["output_root"])
